In [1]:
import pandas as pd
import numpy as np

from dwave.samplers import SimulatedAnnealingSampler
from pyqubo import Binary, LogEncInteger, Placeholder, Constraint
from pathlib import Path


In [2]:
try:
    HERE = Path(__file__)
except NameError:
    HERE = Path().cwd()

DATA = HERE.parent / "data_synthetic"
DATO = HERE.parent / "data_synthetic_out"

DATO.mkdir(exist_ok=True)

WBS_csv = "WBS_Funding_Splits.csv"
CAIS_csv = "CAIS_Deficiencies.csv"
FIM_csv ="FIMS_Master_Scenario_Cost.csv"
QUBO_csv ="Result_QUBO.csv"
FIMS_csv = "FIMS_Inventory.csv"

print(f"HERE: {HERE}")
print(f"DATA: {DATA}")


HERE: /Users/andresrigg/Code/deploy/Experimental_Optimization_Facs/notebooks
DATA: /Users/andresrigg/Code/deploy/Experimental_Optimization_Facs/data_synthetic


In [3]:
FIM_dat = pd.read_csv( (DATA / FIM_csv))
WBS_dat = pd.read_csv((DATA / WBS_csv))
CAIS_sim_dat = pd.read_csv((DATA / CAIS_csv))
FIMS_dat = pd.read_csv((DATA / FIMS_csv))

Project_Type = 'Lab_Type_A'

In [4]:
CAIS_WBS = CAIS_sim_dat.merge(FIMS_dat[['Property_ID','WBS_Element']], on='Property_ID', how='left')
CAIS_WBS_ELMT = CAIS_WBS.merge(WBS_dat[['WBS_Code','Fed_Type_A','Fed_Type_B','Lab_Type_A']], left_on='WBS_Element', right_on='WBS_Code', how='left').rename(columns={'Correction_Cost':'Cost'}).drop(columns={'WBS_Code'})
CAIS_WBS_ID = CAIS_WBS_ELMT[['WBS_Element','Property_ID',Project_Type,'Cost']].groupby(['WBS_Element','Property_ID',Project_Type]).sum().reset_index()

In [5]:
CAIS_WBS_ID['ShareP'] = CAIS_WBS_ID[Project_Type] / 100
CAIS_WBS_ID['ShareS'] = (1 - CAIS_WBS_ID['ShareP'])
CAIS_WBS_ID['cst_P'] = (CAIS_WBS_ID.Cost * CAIS_WBS_ID.ShareP)
CAIS_WBS_ID['cst_S'] = (CAIS_WBS_ID.Cost * CAIS_WBS_ID.ShareS)

In [6]:
CAIS_WBS_ID

,WBS_Element,Property_ID,Lab_Type_A,Cost,ShareP,ShareS,cst_P,cst_S
0,WBS-1.10.01,1938483,100,2219999.0,1.00,0.00,2219999.00,0.00
1,WBS-1.10.01,3094235,100,290834.0,1.00,0.00,290834.00,0.00
2,WBS-1.10.01,4684531,100,3147541.0,1.00,0.00,3147541.00,0.00
3,WBS-1.10.01,5667265,100,2688913.0,1.00,0.00,2688913.00,0.00
4,WBS-1.10.01,6647119,100,2240317.0,1.00,0.00,2240317.00,0.00
5,WBS-1.10.01,7374122,100,2435624.0,1.00,0.00,2435624.00,0.00
6,WBS-1.10.01,8090293,100,939827.0,1.00,0.00,939827.00,0.00
7,WBS-1.10.02,2694522,100,1707590.0,1.00,0.00,1707590.00,0.00
8,WBS-1.10.02,5437923,100,1520519.0,1.00,0.00,1520519.00,0.00
9,WBS-1.10.02,5855124,100,2895298.0,1.00,0.00,2895298.00,0.00


In [7]:
CAIS_WBS_ID[['Property_ID','cst_P','cst_S']]

,Property_ID,cst_P,cst_S
0,1938483,2219999.00,0.00
1,3094235,290834.00,0.00
2,4684531,3147541.00,0.00
3,5667265,2688913.00,0.00
4,6647119,2240317.00,0.00
5,7374122,2435624.00,0.00
6,8090293,939827.00,0.00
7,2694522,1707590.00,0.00
8,5437923,1520519.00,0.00
9,5855124,2895298.00,0.00


In [8]:
x = {index : Binary(f"x_{index}") for index in CAIS_WBS_ID.index}
l = {index : f"x_{index}" for index in CAIS_WBS_ID.index}

CAIS_WBS_ID['Qx'] = x
CAIS_WBS_ID['Lx'] = l

###########################################SHAPES
CAIS_WBS_ID_cols = CAIS_WBS_ID.shape[1]
scenario_cols = CAIS_sim_dat.shape[1]
experiment_count = 5

#COST FUNCTIONS
cst_P = CAIS_WBS_ID['cst_P'] @ CAIS_WBS_ID.Qx
cst_S = CAIS_WBS_ID['cst_S'] @ CAIS_WBS_ID.Qx

#REWARDS
A_P = Placeholder('A_P')
A_S = Placeholder('A_S')

#PENALTIES
M_P = Placeholder('M_P')
M_S = Placeholder('M_S')
lambda_P = Placeholder('lambda_P')
lambda_S = Placeholder('lambda_S')

#PARAMTERS
lim_Bp = 5000000
lim_Bs = 3000000

#PARAMETERSMODEL
feed_dict = {
    'A_P' : -1,
    'A_S' : -1,
    'M_P' : 100,
    'M_S' : 100,
    'lambda_P' : 1,
    'lambda_S' : 1
}

#CONSTRAINT TUNINGS
slk_P = LogEncInteger("slk_P", (0, lim_Bp))
slk_S = LogEncInteger("slk_S", (0, lim_Bs))

P_x = Constraint( ((cst_P + slk_P - lim_Bp)/lim_Bp)**2, label = 'P_x', condition=lambda x: abs(x) < lim_Bp*.025 )
S_x = Constraint( ((cst_S + slk_S - lim_Bs)/lim_Bs)**2, label = 'S_x',  condition=lambda x: abs(x) < lim_Bs*.025 )

In [9]:
E_Qx = A_P*(cst_P/lim_Bp) + A_S*(cst_S/lim_Bs) + M_P*(P_x) + M_S*(S_x) + lambda_P*(slk_P/lim_Bp)**2 + lambda_S*(slk_S/lim_Bs)**2

In [10]:
model = E_Qx.compile()
qubo, offset = model.to_qubo(feed_dict=feed_dict)

In [11]:
qubo

{('x_54', 'x_7'): 5.17542524524,
 ('x_50', 'x_14'): 17.437019084400003,
 ('x_54', 'x_54'): -76.452509254294,
 ('x_8', 'x_3'): 32.708346446776,
 ('x_52', 'x_37'): 41.893348563989996,
 ('slk_P[17]', 'x_40'): 1.877350547456,
 ('slk_S[12]', 'slk_S[12]'): -0.2728783890204444,
 ('x_53', 'x_44'): 37.615349079567,
 ('slk_S[17]', 'slk_S[15]'): 0.0963981548657778,
 ('slk_P[11]', 'x_10'): 0.038278955008,
 ('slk_S[12]', 'x_27'): 0.15696896000000002,
 ('slk_S[16]', 'slk_S[6]'): 9.413882311111113e-05,
 ('x_33', 'x_21'): 4.226695880279999,
 ('slk_S[9]', 'x_28'): 0.0234287104,
 ('slk_P[1]', 'x_32'): 9.37738e-06,
 ('slk_S[9]', 'x_47'): 0.0073566094222222225,
 ('x_35', 'x_33'): 2.7377478539199998,
 ('slk_S[4]', 'x_50'): 0.00019924720000000004,
 ('x_40', 'x_38'): 29.665954309792,
 ('slk_P[18]', 'x_47'): 4.067910746111999,
 ('slk_S[19]', 'x_32'): 20.485199189333336,
 ('slk_P[6]', 'x_4'): 0.0011470423040000001,
 ('slk_S[4]', 'x_51'): 0.0001629706666666667,
 ('x_47', 'x_16'): 32.45095247435556,
 ('slk_P[16]

In [12]:
sampler = SimulatedAnnealingSampler()
sampleset = sampler.sample_qubo(qubo, num_reads=1000)
samplesetFrame = sampleset.to_pandas_dataframe()

In [18]:
samplesetFrame.sort_values('energy')

,slk_P[0],slk_P[10],slk_P[11],slk_P[12],slk_P[13],slk_P[14],slk_P[15],slk_P[16],slk_P[17],slk_P[18],...,x_52,x_53,x_54,x_55,x_6,x_7,x_8,x_9,energy,num_occurrences
656,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,-201.997936,1
568,1,0,1,0,0,1,0,0,0,0,...,0,0,0,0,1,0,0,0,-201.996184,1
461,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,1,0,0,0,-201.994940,1
242,1,0,1,0,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,-201.990957,1
761,0,0,1,1,0,1,0,1,0,0,...,0,0,0,1,0,0,0,0,-201.983852,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
955,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,-199.716107,1
997,0,1,1,1,1,0,1,0,1,0,...,0,0,0,0,1,0,0,0,-199.692913,1
132,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,-199.673391,1
942,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,-199.646099,1


In [14]:
Property_ID_key = CAIS_WBS_ID[['Property_ID','Lx']].merge(samplesetFrame.sort_values('energy',ascending=True).T.add_prefix('EXP'), how='outer' ,left_on='Lx',right_index=True)
CAIS_WBS_ID = CAIS_WBS_ID.merge(samplesetFrame.sort_values('energy',ascending=True).T.add_prefix('EXP'), how='outer' ,left_on='Lx',right_index=True)

In [15]:
scenario_cols = CAIS_WBS_ELMT.shape[1]
experiment_count = 5

In [16]:
QUBOScenario = pd.merge(CAIS_WBS_ELMT, Property_ID_key, how="left", on='Property_ID', ).iloc[0:,0:(scenario_cols + CAIS_WBS_ID_cols + experiment_count )]
QUBOScenario_all = pd.merge(CAIS_WBS_ELMT, Property_ID_key, how="left", on='Property_ID', )
#QUBOScenario_all

In [17]:
CAIS_WBS_ID.to_csv( (DATO / QUBO_csv) )